# Notebook 5 — Full HTLV-1 Protease Analysis Pipeline
## CURE Laboratory: General Chemistry II | HTLV-1 Protease Inhibitor Discovery

**When to complete:** Phase 2, Weeks 9–11 (one run per data-collection week)  
**Team:** ___________________________  
**Inhibitor class:** ___________________________  (Statine / AHPPA / HEA — circle one)  
**Week number:** ___________________________

---

### How to use this notebook

This notebook is your **research instrument**. It runs every analysis your team needs for Phase 2. You do not write new code — you modify clearly marked `# ← CHANGE THIS` lines to point to your team's files and residues.

**Each week in Phase 2 you will:**
1. Upload your new trajectory file to Colab
2. Change the file names in **Section 1.2**
3. Run all cells top-to-bottom (Runtime → Run all)
4. Complete every `ANNOTATION REQUIRED` cell
5. Download the notebook as .ipynb and submit to the LMS

**By the end of Week 11 you will have 3 completed notebooks** — one per simulation run — plus a summary comparison notebook (Section 7).

---

> 🔁 **Pre-computed fallback:** If your trajectory file is not yet available, run the `# FALLBACK` cells to load representative pre-computed data. Label your annotations clearly: *"Pre-computed fallback data used — own trajectory analysis pending."*

---
## Section 1 — Setup and File Loading


### 1.1 Install libraries and import modules


In [ ]:
import subprocess, sys, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib.gridspec import GridSpec

mpl.rcParams['figure.dpi'] = 120
mpl.rcParams['font.family'] = 'sans-serif'
mpl.rcParams['axes.spines.top'] = False
mpl.rcParams['axes.spines.right'] = False

# Install MDAnalysis if not present
try:
    import MDAnalysis as mda
    from MDAnalysis.analysis import rms, align
    from MDAnalysis.analysis.hydrogenbonds import HydrogenBondAnalysis
    MDA_AVAILABLE = True
    print(f"MDAnalysis {mda.__version__} ready")
except ImportError:
    print("Installing MDAnalysis...")
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'MDAnalysis', '--quiet'])
    try:
        import MDAnalysis as mda
        from MDAnalysis.analysis import rms, align
        from MDAnalysis.analysis.hydrogenbonds import HydrogenBondAnalysis
        MDA_AVAILABLE = True
        print("MDAnalysis installed and ready!")
    except ImportError:
        MDA_AVAILABLE = False
        print("MDAnalysis unavailable — fallback data will be used.")

print("All libraries loaded.")


### 1.2 ⚙️ TEAM CONFIGURATION — Edit these values before running

**This is the only section you need to modify.** Change the values between the `# ← CHANGE THIS` comments to match your team's files and assigned inhibitor.


In [ ]:
# ============================================================
# ⚙️  TEAM CONFIGURATION — EDIT THESE VALUES
# ============================================================

# --- File names ---
# Upload your trajectory files to Colab before running this cell
# (Files tab on the left sidebar → Upload)

TOPOLOGY_FILE  = "htlv1_protease_INHIBITOR.pdb"   # ← CHANGE THIS: your .pdb file name
TRAJECTORY_FILE = "htlv1_protease_INHIBITOR.xtc"  # ← CHANGE THIS: your .xtc file name

# --- Inhibitor identity ---
INHIBITOR_NAME  = "LIG"        # ← CHANGE THIS: 3-letter residue name of your inhibitor in the PDB
INHIBITOR_CLASS = "AHPPA"      # ← CHANGE THIS: "Statine", "AHPPA", or "HEA"

# --- Key residues ---
# Polyproline loop residues (DO NOT CHANGE — same for all teams)
LOOP_RESIDUES = list(range(91, 101))   # residues 91–100, both chains

# Catalytic dyad residues (DO NOT CHANGE)
CATALYTIC_ASP = [32, 32]   # Asp32 from each chain (chain A and chain B)

# Residue numbers of active site residues to check for inhibitor contacts
# These are the residues surrounding the binding cleft
ACTIVE_SITE_RESIDUES = [9, 11, 25, 27, 28, 29, 30, 31, 32, 33,
                         48, 49, 50, 51, 52, 96, 97, 98, 99, 100]  # ← adjust if needed

# --- Analysis parameters (standard — only change if instructor advises) ---
HBOND_DIST_CUTOFF  = 3.5    # Angstroms — donor-to-acceptor distance
HBOND_ANGLE_CUTOFF = 120.0  # degrees — D–H···A angle
EQUIL_FRACTION     = 0.20   # Discard first 20% of frames as equilibration
RMSD_STRIDE        = 2      # Analyze every Nth frame (increase for speed if needed)

# ============================================================
print("Team configuration loaded:")
print(f"  Inhibitor class:  {INHIBITOR_CLASS}")
print(f"  Inhibitor name:   {INHIBITOR_NAME}")
print(f"  Topology file:    {TOPOLOGY_FILE}")
print(f"  Trajectory file:  {TRAJECTORY_FILE}")
print(f"  Loop residues:    {LOOP_RESIDUES[0]}–{LOOP_RESIDUES[-1]}")
print(f"  H-bond cutoffs:   {HBOND_DIST_CUTOFF} Å, {HBOND_ANGLE_CUTOFF}°")

# Check whether files exist
topo_ok = os.path.exists(TOPOLOGY_FILE)
traj_ok = os.path.exists(TRAJECTORY_FILE)
LIVE_ANALYSIS = topo_ok and traj_ok and MDA_AVAILABLE
print(f"\nFiles present: topology={topo_ok}, trajectory={traj_ok}")
print(f"Live analysis mode: {LIVE_ANALYSIS}")
if not LIVE_ANALYSIS:
    print("→ Pre-computed fallback data will be used for all analysis.")


### 1.3 Load simulation universe

This cell loads your trajectory into MDAnalysis. It will display basic information about the simulation so you can confirm everything loaded correctly before proceeding.


In [ ]:
if LIVE_ANALYSIS:
    # Load trajectory
    u = mda.Universe(TOPOLOGY_FILE, TRAJECTORY_FILE)

    dt_ps = u.trajectory.dt
    n_frames = u.trajectory.n_frames
    total_ns = (n_frames * dt_ps) / 1000.0
    equil_frame = int(n_frames * EQUIL_FRACTION)

    print(f"Simulation loaded successfully!")
    print(f"  Atoms:              {u.atoms.n_atoms}")
    print(f"  Total frames:       {n_frames}")
    print(f"  Timestep:           {dt_ps:.2f} ps")
    print(f"  Total time:         {total_ns:.2f} ns")
    print(f"  Equilibration end:  frame {equil_frame} ({equil_frame * dt_ps / 1000:.2f} ns)")
    print(f"  Production frames:  {n_frames - equil_frame}")

    # Verify inhibitor is present
    try:
        lig = u.select_atoms(f"resname {INHIBITOR_NAME}")
        print(f"  Inhibitor atoms:    {lig.n_atoms} ({INHIBITOR_NAME})")
        if lig.n_atoms == 0:
            print("  ⚠️  WARNING: No inhibitor atoms found! Check INHIBITOR_NAME in Section 1.2.")
    except Exception as e:
        print(f"  Could not find inhibitor: {e}")

    # Verify loop residues exist
    loop_sel = u.select_atoms(f"resid {' '.join(str(r) for r in LOOP_RESIDUES)} and name CA")
    print(f"  Loop Cα atoms found: {loop_sel.n_atoms} (expected {len(LOOP_RESIDUES)*2} for homodimer)")

else:
    # ── FALLBACK: generate representative pre-computed data ──────────────
    print(f"Loading pre-computed representative data for {INHIBITOR_CLASS} inhibitor...")
    np.random.seed({"Statine": 10, "AHPPA": 20, "HEA": 30}.get(INHIBITOR_CLASS, 42))
    n_frames = 500
    dt_ps = 20.0
    total_ns = n_frames * dt_ps / 1000.0
    equil_frame = int(n_frames * EQUIL_FRACTION)
    print(f"  Simulated frames: {n_frames} ({total_ns:.1f} ns)")
    print(f"  Inhibitor class:  {INHIBITOR_CLASS}")
    print(f"  [This is pre-computed representative data, not your simulation results]")


---
## Section 2 — Global Backbone RMSD

The global RMSD tells you whether the overall protein structure remained stable during the simulation. A stable protein (RMSD plateau around 1–3 Å) means the simulation is behaving correctly and the inhibitor is not causing unphysical distortions.

**Quality control checkpoint:** If global RMSD exceeds ~5 Å, something may be wrong with the simulation setup. Flag this in your annotation and notify the instructor before proceeding.


In [ ]:
# ── LIVE ANALYSIS ──────────────────────────────────────────────────────
if LIVE_ANALYSIS:
    backbone = u.select_atoms("backbone")
    print("Aligning trajectory to reference frame...")
    aligner = align.AlignTraj(u, u, select="backbone", in_memory=True).run()
    print("Calculating global backbone RMSD...")
    R = rms.RMSD(u, select="backbone", ref_frame=0, step=RMSD_STRIDE)
    R.run()
    rmsd_arr  = R.results.rmsd
    times_ns  = rmsd_arr[:, 1] / 1000.0
    rmsd_glob = rmsd_arr[:, 2]

# ── FALLBACK ────────────────────────────────────────────────────────────
else:
    times_ns = np.linspace(0, total_ns, n_frames // RMSD_STRIDE)
    # Inhibitor-class-specific representative RMSD profiles
    profiles = {
        "Statine": {"equil_rmsd": 1.8, "prod_mean": 1.7, "prod_std": 0.15},
        "AHPPA":   {"equil_rmsd": 1.9, "prod_mean": 1.8, "prod_std": 0.18},
        "HEA":     {"equil_rmsd": 2.1, "prod_mean": 2.0, "prod_std": 0.22},
    }
    p = profiles.get(INHIBITOR_CLASS, profiles["AHPPA"])
    equil = p["equil_rmsd"] * (1 - np.exp(-times_ns / (total_ns * 0.2)))
    noise = np.random.normal(0, p["prod_std"], len(times_ns))
    rmsd_glob = equil + noise
    rmsd_glob = np.clip(rmsd_glob, 0.05, 5.0)
    print(f"Pre-computed global RMSD loaded ({INHIBITOR_CLASS})")

# Production phase values
prod_mask  = times_ns >= (total_ns * EQUIL_FRACTION)
prod_rmsd  = rmsd_glob[prod_mask]
prod_mean  = prod_rmsd.mean()
prod_std   = prod_rmsd.std()

print(f"Global backbone RMSD — Production phase:")
print(f"  Mean: {prod_mean:.3f} Å")
print(f"  SD:   {prod_std:.3f} Å")
print(f"  Range: {prod_rmsd.min():.2f} – {prod_rmsd.max():.2f} Å")
print(f"  Status: {'✓ STABLE' if prod_mean < 3.5 else '⚠ HIGH — review simulation'}")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

# --- RMSD time series ---
ax = axes[0]
equil_end_time = total_ns * EQUIL_FRACTION
ax.axvspan(0, equil_end_time, alpha=0.10, color='#CC3300', label='Equilibration')
ax.axvspan(equil_end_time, total_ns, alpha=0.06, color='#2E5EAA', label='Production')
ax.plot(times_ns, rmsd_glob, color='#2E5EAA', linewidth=1.0, alpha=0.75)

# Smoothed trend
win = max(5, len(rmsd_glob) // 20)
smoothed = pd.Series(rmsd_glob).rolling(window=win, center=True).mean()
ax.plot(times_ns, smoothed, color='#CC6600', linewidth=2.2, label='Smoothed trend')

ax.axhline(prod_mean, color='gray', linestyle='--', alpha=0.8,
           label=f'Prod. mean: {prod_mean:.2f} Å')
ax.set_xlabel('Simulation Time (ns)', fontsize=12)
ax.set_ylabel('RMSD (Å)', fontsize=12)
ax.set_title(f'Global Backbone RMSD\n{INHIBITOR_CLASS}-bound HTLV-1 Protease', fontweight='bold')
ax.legend(fontsize=9)
ax.set_xlim(0, total_ns)
ax.set_ylim(bottom=0)
ax.grid(True, alpha=0.22)

# --- Distribution ---
ax2 = axes[1]
ax2.hist(prod_rmsd, bins=30, color='#2E5EAA', alpha=0.8, edgecolor='white')
ax2.axvline(prod_mean, color='#CC6600', linewidth=2.0, label=f'Mean: {prod_mean:.2f} Å')
ax2.axvline(prod_mean + prod_std, color='gray', linestyle=':', linewidth=1.5)
ax2.axvline(prod_mean - prod_std, color='gray', linestyle=':', linewidth=1.5,
            label=f'±1 SD: {prod_std:.2f} Å')
ax2.set_xlabel('RMSD (Å)', fontsize=12)
ax2.set_ylabel('Frame Count', fontsize=12)
ax2.set_title('RMSD Distribution (Production Phase)', fontweight='bold')
ax2.legend(fontsize=9)
ax2.grid(True, alpha=0.22)

plt.suptitle(f'Team: {INHIBITOR_CLASS} Inhibitor  |  Quality Check: {"✓ PASS" if prod_mean < 3.5 else "⚠ REVIEW"}',
             fontsize=11, y=1.02)
plt.tight_layout()
fname = f'S2_global_rmsd_{INHIBITOR_CLASS}.png'
plt.savefig(fname, dpi=150, bbox_inches='tight')
plt.show()
print(f"Figure saved: {fname}")


### ✏️ ANNOTATION REQUIRED — Section 2

**Answer all three points before moving on:**

1. **Equilibration:** At what time (ns) does the RMSD appear to plateau? Mark this on the plot by adding a vertical line (optional — change `equil_end_time` in the config if needed). Is the equilibration fraction (20%) you used appropriate for this simulation?

2. **Stability assessment:** Is the global backbone RMSD stable during the production phase? Does the protein appear to remain in a near-native conformation or undergo a major structural change? How does this compare to the lysozyme dress rehearsal (Notebook 3)?

3. **Quality check:** Does the output show "✓ STABLE" or "⚠ REVIEW"? If review, describe what you observe and what you will do before proceeding (consult instructor, check simulation setup, use backup trajectory).


### My Annotation — Section 2:

*[YOUR ANSWER HERE — one paragraph per point, complete sentences]*


---
## Section 3 — Polyproline Loop RMSD (Your Primary Research Metric)

This is the core of your research. The polyproline loop RMSD tells you how much the loop (residues 91–100) moves during simulation relative to its crystal structure position. Compare this value to your hypothesis prediction.

**Expected range:** The apo (unbound) loop typically shows higher RMSD (2–5 Å) than inhibitor-bound systems. Whether your inhibitor restricts loop motion — and by how much — is what you are discovering.


In [ ]:
# ── LIVE ANALYSIS ──────────────────────────────────────────────────────
if LIVE_ANALYSIS:
    loop_sel_str = "resid " + " ".join(str(r) for r in LOOP_RESIDUES) + " and backbone"
    loop_atoms = u.select_atoms(loop_sel_str)
    print(f"Loop selection: {loop_atoms.n_atoms} backbone atoms")

    R_loop = rms.RMSD(u, select=loop_sel_str, ref_frame=0, step=RMSD_STRIDE)
    R_loop.run()
    loop_rmsd_arr = R_loop.results.rmsd
    times_ns_l    = loop_rmsd_arr[:, 1] / 1000.0
    loop_rmsd     = loop_rmsd_arr[:, 2]

# ── FALLBACK ────────────────────────────────────────────────────────────
else:
    times_ns_l = times_ns.copy()
    # Inhibitor-specific loop RMSD profiles (based on plausible research outcomes)
    loop_profiles = {
        "Statine": {"mean": 1.6, "std": 0.25, "seed": 11},
        "AHPPA":   {"mean": 1.4, "std": 0.20, "seed": 21},
        "HEA":     {"mean": 2.4, "std": 0.40, "seed": 31},
    }
    lp = loop_profiles.get(INHIBITOR_CLASS, loop_profiles["AHPPA"])
    np.random.seed(lp["seed"])
    equil_phase = lp["mean"] * (1 - np.exp(-times_ns_l / (total_ns * 0.15)))
    noise = np.random.normal(0, lp["std"], len(times_ns_l))
    slow_osc = 0.3 * np.sin(times_ns_l * 1.5)
    loop_rmsd = equil_phase + noise + slow_osc
    loop_rmsd = np.clip(loop_rmsd, 0.1, 6.0)
    print(f"Pre-computed loop RMSD loaded ({INHIBITOR_CLASS})")

prod_mask_l     = times_ns_l >= (total_ns * EQUIL_FRACTION)
loop_prod       = loop_rmsd[prod_mask_l]
loop_mean       = loop_prod.mean()
loop_std        = loop_prod.std()
loop_max        = loop_prod.max()

print(f"\nPolyproline Loop RMSD — Production Phase:")
print(f"  Mean RMSD: {loop_mean:.3f} Å")
print(f"  SD:        {loop_std:.3f} Å")
print(f"  Max RMSD:  {loop_max:.3f} Å")
print(f"  Flexibility interpretation: {'High (>2.5 Å)' if loop_mean > 2.5 else 'Moderate (1.5–2.5 Å)' if loop_mean > 1.5 else 'Low (<1.5 Å)'}")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

# --- Loop RMSD time series ---
ax = axes[0]
equil_end_t = total_ns * EQUIL_FRACTION
ax.axvspan(0, equil_end_t, alpha=0.10, color='#CC3300')
ax.axvspan(equil_end_t, total_ns, alpha=0.06, color='#2E5EAA')
ax.plot(times_ns_l, loop_rmsd, color='#9B59B6', linewidth=1.0, alpha=0.75)

# Smoothed
win = max(5, len(loop_rmsd) // 20)
loop_smooth = pd.Series(loop_rmsd).rolling(window=win, center=True).mean()
ax.plot(times_ns_l, loop_smooth, color='#5E3B8E', linewidth=2.2, label='Smoothed')

ax.axhline(loop_mean, color='gray', linestyle='--',
           label=f'Mean: {loop_mean:.2f} Å')
ax.fill_between(times_ns_l[prod_mask_l],
                loop_mean - loop_std, loop_mean + loop_std,
                alpha=0.12, color='#9B59B6', label=f'±1 SD: {loop_std:.2f} Å')

ax.set_xlabel('Simulation Time (ns)', fontsize=12)
ax.set_ylabel('Loop RMSD (Å)', fontsize=12)
ax.set_title(f'Polyproline Loop RMSD (Res 91–100)\n{INHIBITOR_CLASS}-bound Protease', fontweight='bold')
ax.legend(fontsize=9)
ax.set_xlim(0, total_ns)
ax.set_ylim(bottom=0)
ax.grid(True, alpha=0.22)

# --- Comparison bar: loop vs global RMSD ---
ax2 = axes[1]
categories = ['Global\nBackbone', 'Polyproline\nLoop (91–100)']
means_   = [prod_mean, loop_mean]
stds_    = [prod_std,  loop_std]
colors_  = ['#2E5EAA', '#9B59B6']
bars = ax2.bar(categories, means_, color=colors_, alpha=0.85,
               yerr=stds_, capsize=8, edgecolor='white', linewidth=0.8)
ax2.set_ylabel('Mean RMSD (Å)', fontsize=12)
ax2.set_title(f'RMSD Comparison\n{INHIBITOR_CLASS} Inhibitor', fontweight='bold')
ax2.grid(True, axis='y', alpha=0.22)

# Add value labels on bars
for bar, mean, std in zip(bars, means_, stds_):
    ax2.text(bar.get_x() + bar.get_width()/2, mean + std + 0.05,
             f'{mean:.2f} Å', ha='center', va='bottom', fontsize=11, fontweight='bold')

plt.tight_layout()
fname = f'S3_loop_rmsd_{INHIBITOR_CLASS}.png'
plt.savefig(fname, dpi=150, bbox_inches='tight')
plt.show()
print(f"Figure saved: {fname}")

# Store for cross-team comparison (Section 7)
LOOP_RMSD_MEAN = loop_mean
LOOP_RMSD_STD  = loop_std


### ✏️ ANNOTATION REQUIRED — Section 3

**This is your most important annotation. Be thorough.**

1. **Compare to your hypothesis:** In your research proposal, you predicted that the loop RMSD would be [higher/lower] than the apo system and would have a mean around [X] Å. Does your result match this prediction? Quote your original hypothesis and your measured mean RMSD.

2. **Flexibility interpretation:** Based on the loop RMSD value, characterize the loop as flexible, moderately flexible, or restricted. What does this mean for how the inhibitor is affecting the protease structure?

3. **Compare global vs. loop:** Is the loop RMSD higher or lower than the global backbone RMSD? What does this difference tell you about where in the protein structure the most movement is occurring? Is this consistent with what you expected based on the Protease Primer (Supplement S1)?

4. **Thermodynamic connection:** If a more restricted loop (lower RMSD) correlates with better binding, what does this imply about the ΔS contribution to binding? Is entropy favorable or unfavorable? How does this connect to the ΔG = ΔH − TΔS equation from lecture?


### My Annotation — Section 3:

*[YOUR ANSWER HERE — minimum 2 sentences per point, 8 sentences total]*


---
## Section 4 — Per-Residue Flexibility of the Loop (RMSF)

RMSF gives you the flexibility of each individual residue within the loop. This tells you *which specific proline and surrounding residues* are most affected by your inhibitor — a more detailed picture than the single loop RMSD value.


In [ ]:
if LIVE_ANALYSIS:
    from MDAnalysis.analysis import rms
    u3 = mda.Universe(TOPOLOGY_FILE, TRAJECTORY_FILE)
    prod_start = int(u3.trajectory.n_frames * EQUIL_FRACTION)
    loop_ca = u3.select_atoms(
        "resid " + " ".join(str(r) for r in LOOP_RESIDUES) + " and name CA and segid A"
    )
    R_rmsf = rms.RMSF(loop_ca).run(start=prod_start)
    loop_resids = loop_ca.resids
    loop_rmsf   = R_rmsf.results.rmsf
    RMSF_SRC = "live"
else:
    loop_resids = np.array(LOOP_RESIDUES)
    rmsf_profiles = {
        "Statine": [0.9, 1.1, 1.3, 1.6, 1.7, 1.5, 1.3, 1.1, 0.9, 0.8],
        "AHPPA":   [0.7, 0.9, 1.0, 1.2, 1.3, 1.2, 1.0, 0.8, 0.7, 0.6],
        "HEA":     [1.2, 1.5, 1.9, 2.4, 2.6, 2.5, 2.2, 1.8, 1.4, 1.1],
    }
    base = np.array(rmsf_profiles.get(INHIBITOR_CLASS, rmsf_profiles["AHPPA"]))
    noise = np.random.normal(0, 0.06, len(base))
    loop_rmsf = np.clip(base + noise, 0.1, 4.0)
    RMSF_SRC = "precomputed"
    print(f"Pre-computed RMSF loaded ({INHIBITOR_CLASS})")

print("Per-residue RMSF for polyproline loop:")
for res, rmsf_val in zip(loop_resids, loop_rmsf):
    bar = '█' * int(rmsf_val * 10)
    print(f"  Residue {res:3d}: {rmsf_val:.3f} Å  {bar}")
print(f"\nMost flexible residue: {loop_resids[loop_rmsf.argmax()]} ({loop_rmsf.max():.2f} Å)")
print(f"Most rigid residue:     {loop_resids[loop_rmsf.argmin()]} ({loop_rmsf.min():.2f} Å)")


In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))

bar_colors = ['#9B59B6' if v > np.mean(loop_rmsf) + np.std(loop_rmsf)
              else '#2E5EAA' for v in loop_rmsf]
bars = ax.bar(loop_resids, loop_rmsf, color=bar_colors, alpha=0.85, edgecolor='white')

ax.axhline(np.mean(loop_rmsf), color='gray', linestyle='--',
           label=f'Loop mean RMSF ({np.mean(loop_rmsf):.2f} Å)')
ax.axhline(np.mean(loop_rmsf) + np.std(loop_rmsf), color='#9B59B6',
           linestyle=':', alpha=0.7, label=f'Mean + 1 SD')

# Label each bar
for bar, resid, val in zip(bars, loop_resids, loop_rmsf):
    ax.text(bar.get_x() + bar.get_width()/2, val + 0.03,
            str(resid), ha='center', va='bottom', fontsize=8, rotation=0)

ax.set_xlabel('Residue Number (Polyproline Loop)', fontsize=12)
ax.set_ylabel('RMSF (Å)', fontsize=12)
ax.set_title(f'Per-Residue Flexibility of Polyproline Loop\n{INHIBITOR_CLASS}-bound HTLV-1 Protease',
             fontweight='bold')
ax.set_xlim(loop_resids[0] - 0.7, loop_resids[-1] + 0.7)
ax.set_ylim(bottom=0)
ax.legend(fontsize=10)
ax.grid(True, axis='y', alpha=0.22)

plt.tight_layout()
fname = f'S4_loop_rmsf_{INHIBITOR_CLASS}.png'
plt.savefig(fname, dpi=150, bbox_inches='tight')
plt.show()
print(f"Figure saved: {fname}")


### ✏️ ANNOTATION REQUIRED — Section 4

1. **Hotspot residues:** Which residues within the loop have the highest RMSF? Are they at the ends of the loop, in the middle, or scattered? Does this pattern make sense given the loop's structure (proline residues are more rigid than most)?

2. **Inhibitor contact logic:** If an inhibitor forms a hydrogen bond with a specific loop residue, you would expect that residue's RMSF to be *lower* than surrounding residues (the H-bond pins it down). Look at your H-bond results in Section 5 — do the residues with H-bond contacts show lower RMSF?

3. **Compare to expected:** Based on the Protease Primer (Supplement S1), the loop spans residues 91–100. Did any residues outside this range show unexpectedly high flexibility? If so, what does this suggest?


### My Annotation — Section 4:

*[YOUR ANSWER HERE]*


---
## Section 5 — Hydrogen Bond Occupancy: Inhibitor–Protease Interactions

This section identifies every hydrogen bond between your inhibitor and the protease, and calculates what fraction of the simulation time each H-bond is present. These values are your direct evidence for the molecular basis of inhibitor binding.


In [ ]:
if LIVE_ANALYSIS:
    u4 = mda.Universe(TOPOLOGY_FILE, TRAJECTORY_FILE)
    prod_start4 = int(u4.trajectory.n_frames * EQUIL_FRACTION)

    inhib_sel = f"resname {INHIBITOR_NAME}"
    protease_sel = f"protein and not resname {INHIBITOR_NAME}"

    print("Running H-bond analysis (inhibitor ↔ protease)...")
    hba = HydrogenBondAnalysis(
        universe=u4,
        donors_sel=None,
        acceptors_sel=None,
        d_a_cutoff=HBOND_DIST_CUTOFF,
        d_h_a_angle_cutoff=HBOND_ANGLE_CUTOFF,
        between=[[inhib_sel, protease_sel]],
        update_selections=False
    )
    hba.run(start=prod_start4)

    total_prod = u4.trajectory.n_frames - prod_start4
    pair_counts = {}
    for row in hba.results.hbonds:
        key = (int(row[1]), int(row[3]), row[4].strip(), row[6].strip())
        pair_counts[key] = pair_counts.get(key, 0) + 1

    hb_records = []
    for (d_resid, a_resid, d_atom, a_atom), count in pair_counts.items():
        occ = round(count / total_prod * 100, 1)
        hb_records.append({'donor_res': d_resid, 'acceptor_res': a_resid,
                            'donor_atom': d_atom, 'acceptor_atom': a_atom,
                            'occupancy_%': occ, 'frames': count})
    hb_df = pd.DataFrame(hb_records).sort_values('occupancy_%', ascending=False)
    HB_SRC = "live"

else:
    # Pre-computed H-bond occupancy representative data
    hb_templates = {
        "Statine": [
            {"donor_res": 32,  "acceptor_res": "LIG", "donor_atom": "OD1", "acceptor_atom": "O3", "occupancy_%": 84.2},
            {"donor_res": "LIG","acceptor_res": 32,   "donor_atom": "O3H", "acceptor_atom": "OD2","occupancy_%": 78.5},
            {"donor_res": 95,  "acceptor_res": "LIG", "donor_atom": "N",   "acceptor_atom": "O1", "occupancy_%": 52.1},
            {"donor_res": "LIG","acceptor_res": 97,   "donor_atom": "N1H", "acceptor_atom": "O",  "occupancy_%": 41.8},
            {"donor_res": 29,  "acceptor_res": "LIG", "donor_atom": "N",   "acceptor_atom": "O2", "occupancy_%": 23.4},
        ],
        "AHPPA": [
            {"donor_res": 32,  "acceptor_res": "LIG", "donor_atom": "OD1", "acceptor_atom": "OH", "occupancy_%": 91.3},
            {"donor_res": "LIG","acceptor_res": 32,   "donor_atom": "OH",  "acceptor_atom": "OD2","occupancy_%": 87.6},
            {"donor_res": 98,  "acceptor_res": "LIG", "donor_atom": "N",   "acceptor_atom": "O",  "occupancy_%": 62.4},
            {"donor_res": "LIG","acceptor_res": 50,   "donor_atom": "NH",  "acceptor_atom": "O",  "occupancy_%": 38.2},
            {"donor_res": 96,  "acceptor_res": "LIG", "donor_atom": "O",   "acceptor_atom": "N1", "occupancy_%": 28.9},
        ],
        "HEA": [
            {"donor_res": 32,  "acceptor_res": "LIG", "donor_atom": "OD1", "acceptor_atom": "OH", "occupancy_%": 71.8},
            {"donor_res": "LIG","acceptor_res": 32,   "donor_atom": "OH",  "acceptor_atom": "OD2","occupancy_%": 65.3},
            {"donor_res": "LIG","acceptor_res": 93,   "donor_atom": "NH2", "acceptor_atom": "O",  "occupancy_%": 34.7},
            {"donor_res": 50,  "acceptor_res": "LIG", "donor_atom": "NH",  "acceptor_atom": "N",  "occupancy_%": 22.1},
            {"donor_res": 97,  "acceptor_res": "LIG", "donor_atom": "O",   "acceptor_atom": "OH", "occupancy_%": 14.6},
        ],
    }
    rows = hb_templates.get(INHIBITOR_CLASS, hb_templates["AHPPA"])
    hb_df = pd.DataFrame(rows).sort_values('occupancy_%', ascending=False)
    HB_SRC = "precomputed"
    print(f"Pre-computed H-bond data loaded ({INHIBITOR_CLASS})")

print("\nInhibitor–Protease Hydrogen Bonds:")
print(hb_df.to_string(index=False))
print(f"\nTotal H-bonds detected: {len(hb_df)}")
print(f"Strong (>70%): {(hb_df['occupancy_%'] > 70).sum()}")
print(f"Moderate (30–70%): {((hb_df['occupancy_%'] > 30) & (hb_df['occupancy_%'] <= 70)).sum()}")
print(f"Weak (<30%): {(hb_df['occupancy_%'] <= 30).sum()}")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Horizontal bar chart: H-bond occupancy ---
ax1 = axes[0]
n_hb = len(hb_df)
y_pos = range(n_hb)
labels = []
for _, row in hb_df.iterrows():
    d = f"Res{row['donor_res']}:{row['donor_atom']}"
    a = f"Res{row['acceptor_res']}:{row['acceptor_atom']}"
    labels.append(f"{d} → {a}")

bar_cols = ['#27AE60' if o > 70 else ('#E8A838' if o > 30 else '#CC3300')
            for o in hb_df['occupancy_%']]
ax1.barh(list(y_pos), hb_df['occupancy_%'].values,
         color=bar_cols, alpha=0.88, edgecolor='white')
ax1.set_yticks(list(y_pos))
ax1.set_yticklabels(labels, fontsize=9)
ax1.set_xlabel('Occupancy (%)', fontsize=11)
ax1.set_title(f'Inhibitor–Protease H-Bond Occupancy\n{INHIBITOR_CLASS}', fontweight='bold')
ax1.axvline(70, color='#27AE60', linestyle='--', alpha=0.7, label='Strong (70%)')
ax1.axvline(30, color='#CC3300', linestyle='--', alpha=0.7, label='Weak (30%)')
ax1.set_xlim(0, 108)
ax1.legend(fontsize=9)
ax1.grid(True, axis='x', alpha=0.22)
for i, occ in enumerate(hb_df['occupancy_%']):
    ax1.text(occ + 1, i, f'{occ:.1f}%', va='center', fontsize=8.5)

# --- Loop H-bond summary ---
ax2 = axes[1]
loop_range = range(LOOP_RESIDUES[0], LOOP_RESIDUES[-1]+1)
loop_hbs = hb_df[hb_df['acceptor_res'].astype(str).str.isnumeric() &
                  hb_df['acceptor_res'].astype(str).apply(
                      lambda x: int(x) in loop_range if x.isnumeric() else False)]
loop_hbs2 = hb_df[hb_df['donor_res'].astype(str).str.isnumeric() &
                   hb_df['donor_res'].astype(str).apply(
                       lambda x: int(x) in loop_range if x.isnumeric() else False)]
all_loop_hbs = pd.concat([loop_hbs, loop_hbs2]).drop_duplicates()

if len(all_loop_hbs) > 0:
    lh_labels = [f"Res{row['donor_res']} → Res{row['acceptor_res']}"
                 for _, row in all_loop_hbs.iterrows()]
    lh_colors = ['#9B59B6' if o > 50 else '#C39BD3'
                 for o in all_loop_hbs['occupancy_%']]
    ax2.bar(range(len(all_loop_hbs)), all_loop_hbs['occupancy_%'].values,
            color=lh_colors, alpha=0.85, edgecolor='white')
    ax2.set_xticks(range(len(all_loop_hbs)))
    ax2.set_xticklabels(lh_labels, rotation=30, ha='right', fontsize=9)
    ax2.set_ylabel('Occupancy (%)', fontsize=11)
    ax2.set_title(f'Loop-Specific H-Bonds (Res 91–100)\n{INHIBITOR_CLASS}', fontweight='bold')
    ax2.axhline(50, color='gray', linestyle='--', alpha=0.7)
    ax2.grid(True, axis='y', alpha=0.22)
else:
    ax2.text(0.5, 0.5, f'No direct H-bonds detected\nbetween {INHIBITOR_CLASS}\nand loop residues 91–100',
             ha='center', va='center', transform=ax2.transAxes, fontsize=12,
             bbox=dict(boxstyle='round', facecolor='#F0F0F0'))
    ax2.set_title(f'Loop-Specific H-Bonds (Res 91–100)\n{INHIBITOR_CLASS}', fontweight='bold')

plt.tight_layout()
fname = f'S5_hbonds_{INHIBITOR_CLASS}.png'
plt.savefig(fname, dpi=150, bbox_inches='tight')
plt.show()
print(f"Figure saved: {fname}")
print(f"\nLoop H-bonds detected: {len(all_loop_hbs)}")

# Store for cross-team comparison
STRONGEST_HBOND_OCC = hb_df['occupancy_%'].max() if len(hb_df) > 0 else 0.0
LOOP_HBOND_OCC = all_loop_hbs['occupancy_%'].max() if len(all_loop_hbs) > 0 else 0.0


### ✏️ ANNOTATION REQUIRED — Section 5

1. **Catalytic dyad contacts:** Do you observe H-bonds with Asp32 (from either chain)? What is the occupancy? The catalytic aspartates are the primary binding anchors for all three inhibitor classes — high occupancy here indicates the inhibitor is staying in the active site.

2. **Loop contacts:** Are there any direct H-bonds between your inhibitor and the polyproline loop residues (91–100)? If yes, which residues and what occupancy? If no, does the absence of direct H-bonds rule out indirect effects on loop dynamics? Explain.

3. **Thermodynamic interpretation:** Pick the strongest H-bond (highest occupancy) and calculate its approximate ΔG using ΔG = −RT ln K, where K = occupancy/(1−occupancy) and T = 300 K, R = 8.314 J/mol·K. Is this a thermodynamically favorable interaction?

4. **Connection to inhibitor class chemistry:** Do the H-bond patterns match what you predicted in your research proposal, based on the functional groups of your inhibitor class (from Supplement S1, Part 6)?


### My Annotation — Section 5:

*[YOUR ANSWER HERE — minimum 2 sentences per point]*


---
## Section 6 — Summary Table for This Run

Run this cell after completing Sections 2–5. It generates a formatted summary table of all key metrics for this trajectory run, ready to copy into your manuscript.


In [ ]:
summary = {
    "Inhibitor class":              INHIBITOR_CLASS,
    "Trajectory file":              TRAJECTORY_FILE,
    "Total simulation time (ns)":   f"{total_ns:.1f}",
    "Production frames analyzed":   str(int(n_frames * (1 - EQUIL_FRACTION))),
    "Global backbone RMSD — mean (Å)": f"{prod_mean:.3f} ± {prod_std:.3f}",
    "Loop RMSD — mean (Å)":         f"{LOOP_RMSD_MEAN:.3f} ± {LOOP_RMSD_STD:.3f}",
    "Loop RMSD — max (Å)":          f"{loop_prod.max():.3f}",
    "Most flexible loop residue":   str(loop_resids[loop_rmsf.argmax()]),
    "H-bond cutoffs used":          f"{HBOND_DIST_CUTOFF} Å, {HBOND_ANGLE_CUTOFF}°",
    "Total inhibitor–protease H-bonds": str(len(hb_df)),
    "Strongest H-bond occupancy (%)":   f"{STRONGEST_HBOND_OCC:.1f}",
    "Loop H-bond occupancy — max (%)":  f"{LOOP_HBOND_OCC:.1f}",
    "Data source":                  "Live analysis" if LIVE_ANALYSIS else "Pre-computed fallback",
}

print("=" * 60)
print(f"  RESULTS SUMMARY — {INHIBITOR_CLASS} Inhibitor")
print("=" * 60)
for key, val in summary.items():
    print(f"  {key:<42} {val}")
print("=" * 60)
print("\nShare this table with your teammates and instructor.")
print("Copy these values into Table 1 of your research manuscript.")


---
## Section 7 — Cross-Team Comparison (Complete in Week 12, Phase 3)

**Instructions for Week 12:** Once all three teams have completed their simulations and shared their summary tables, enter the class-wide data in the cells below to generate the comparison plots for your manuscript Figure 2.

Each team enters their own values in the config cell. The team that runs this section last fills in all three columns.


In [ ]:
# ── FILL IN AFTER ALL TEAMS SHARE DATA ──────────────────────────────────
# Replace the placeholder values with actual results from each team's summary table

team_data = pd.DataFrame({
    'Inhibitor':    ['Statine',  'AHPPA',    'HEA'],
    'Loop_RMSD':    [1.6,        1.4,        2.4],    # ← REPLACE with actual values
    'Loop_RMSD_SD': [0.25,       0.20,       0.40],   # ← REPLACE with actual SDs
    'Max_HB_Occ':   [84.2,       91.3,       71.8],   # ← REPLACE with actual values
    'Loop_HB_Occ':  [52.1,       62.4,       34.7],   # ← REPLACE with actual values
    'N_HBonds':     [5,          5,          5],      # ← REPLACE with actual counts
})

print("Cross-team data table:")
print(team_data.to_string(index=False))
print("\n⚠️  If values still show placeholder defaults, update team_data above before plotting.")


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
colors = ['#2E5EAA', '#27AE60', '#CC6600']

# --- Panel A: Loop RMSD comparison ---
ax1 = axes[0]
bars = ax1.bar(team_data['Inhibitor'], team_data['Loop_RMSD'],
               color=colors, alpha=0.85, edgecolor='white',
               yerr=team_data['Loop_RMSD_SD'], capsize=7)
ax1.set_ylabel('Mean Loop RMSD (Å)', fontsize=12)
ax1.set_title('A. Polyproline Loop Flexibility\n(Lower = More Restricted)',
              fontweight='bold', fontsize=11)
ax1.grid(True, axis='y', alpha=0.22)
for bar, val in zip(bars, team_data['Loop_RMSD']):
    ax1.text(bar.get_x() + bar.get_width()/2, val + 0.05,
             f'{val:.2f}', ha='center', fontsize=11, fontweight='bold')

# --- Panel B: Strongest H-bond occupancy ---
ax2 = axes[1]
bars2 = ax2.bar(team_data['Inhibitor'], team_data['Max_HB_Occ'],
                color=colors, alpha=0.85, edgecolor='white')
ax2.axhline(70, color='gray', linestyle='--', alpha=0.7, label='Strong H-bond threshold')
ax2.set_ylabel('Occupancy (%)', fontsize=12)
ax2.set_title('B. Strongest H-Bond Occupancy\n(Higher = More Persistent)',
              fontweight='bold', fontsize=11)
ax2.set_ylim(0, 110)
ax2.legend(fontsize=9)
ax2.grid(True, axis='y', alpha=0.22)
for bar, val in zip(bars2, team_data['Max_HB_Occ']):
    ax2.text(bar.get_x() + bar.get_width()/2, val + 1.2,
             f'{val:.1f}%', ha='center', fontsize=11, fontweight='bold')

# --- Panel C: Loop-specific H-bond occupancy ---
ax3 = axes[2]
bars3 = ax3.bar(team_data['Inhibitor'], team_data['Loop_HB_Occ'],
                color=colors, alpha=0.85, edgecolor='white')
ax3.axhline(30, color='#CC3300', linestyle='--', alpha=0.7, label='Weak/moderate threshold')
ax3.set_ylabel('Loop H-Bond Occupancy (%)', fontsize=12)
ax3.set_title('C. H-Bond to Loop Residues 91–100\n(Correlates with Loop Restriction?)',
              fontweight='bold', fontsize=11)
ax3.set_ylim(0, 110)
ax3.legend(fontsize=9)
ax3.grid(True, axis='y', alpha=0.22)
for bar, val in zip(bars3, team_data['Loop_HB_Occ']):
    ax3.text(bar.get_x() + bar.get_width()/2, val + 1.2,
             f'{val:.1f}%', ha='center', fontsize=11, fontweight='bold')

plt.suptitle('Cross-Team Comparison: HTLV-1 Protease Inhibitor Analysis',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
fname = 'S7_cross_team_comparison.png'
plt.savefig(fname, dpi=150, bbox_inches='tight')
plt.show()
print(f"Figure saved: {fname}")
print("This figure can be used directly as Figure 2 in your manuscript.")


### ✏️ ANNOTATION REQUIRED — Section 7 (complete in Week 12)

1. **Overall pattern:** Looking at Panel A, which inhibitor class shows the most restricted polyproline loop? Which shows the most flexible? Does this ranking match your team's original hypothesis from Week 8?

2. **H-bond correlation:** Is there a correlation between loop H-bond occupancy (Panel C) and loop RMSD (Panel A)? In other words, do inhibitors that form more persistent H-bonds with loop residues also show more restricted loop motion? What does this tell you about the mechanism of loop restriction?

3. **Ranking and drug design implications:** If you were advising a medicinal chemist trying to design a better HTLV-1 protease inhibitor, which scaffold (statine, AHPPA, or HEA) would you recommend as a starting point based on your class's data? Justify your recommendation using the data from all three panels.


### My Annotation — Section 7:

*[Complete this after all teams share data in Week 12 — minimum 3 sentences per point]*


---
## ✅ Notebook 5 Weekly Completion Checklist

Complete this for **each trajectory run** (Weeks 9, 10, 11):

**Before running:**
- [ ] Team configuration (Section 1.2) updated with correct file names and inhibitor name
- [ ] Trajectory file uploaded to Colab

**After running:**
- [ ] Section 2: Global RMSD — quality check passes (✓ STABLE shown)
- [ ] Section 3: Loop RMSD — production mean recorded in team log
- [ ] Section 4: Per-residue RMSF — most flexible residue identified
- [ ] Section 5: H-bond occupancy table — strongest H-bond and loop H-bonds recorded
- [ ] Section 6: Summary table — printed and copied to team shared document
- [ ] All ANNOTATION REQUIRED cells completed with full sentences
- [ ] Figures saved (4 .png files in Colab files panel)
- [ ] Notebook downloaded as .ipynb and submitted to LMS

**For Section 7 (Week 12 only):**
- [ ] All three teams' summary table values entered in team_data
- [ ] Cross-team comparison figure generated and saved
- [ ] Section 7 annotation completed

---
> **Questions?** Contact Dr. Manukyan at amanukyan@hostos.cuny.edu or visit office hours.  
> Include your team name and which section/cell is causing the issue in your email.
